In [2]:
%load_ext autoreload
%autoreload 2

# Joining Land Registry Price Data with Energy Performance Certificates (2015 - 2025)

In [3]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [4]:
from src.cleaning import load
from src.cleaning import config

In [5]:

pp = load.read_price_paid(config.DATA_PATH, config.PP_FILENAME)
pp.shape[0]

135557

In [6]:
epc = load.read_epc(config.DATA_PATH, config.EPC_FILENAME, config.EPC_COLS)
epc.info()

<class 'pandas.DataFrame'>
RangeIndex: 216054 entries, 0 to 216053
Data columns (total 28 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   certificate_number           216054 non-null  str           
 1   address1                     216054 non-null  str           
 2   address2                     117532 non-null  str           
 3   address3                     27304 non-null   str           
 4   postcode                     216054 non-null  str           
 5   posttown                     213327 non-null  str           
 6   address                      216054 non-null  str           
 7   local_authority              216054 non-null  str           
 8   local_authority_label        216054 non-null  str           
 9   built_form                   213548 non-null  str           
 10  construction_age_band        216054 non-null  str           
 11  current_energy_efficiency    216054 n

Correct row number and dtypes

In [7]:
epc = load.filter_local_authorities(epc, config.TARGET_LOCAL_AUTHORITIES)

epc.shape

(216050, 28)

4 rows dropped as expected

In [8]:
from src.cleaning import normalise

epc = normalise.strip_whitespace(epc)
epc = normalise.remove_commas(epc, config.EPC_ADDRESS_COLS)
epc['address1']

0                  10 Newman Road
1                 10 Barlow Close
2                65 Campbell Road
3           12 St. Michaels Close
4         Flat 10 Canalside House
                   ...           
216049               30 Appletons
216050        1d Walton Well Road
216051            8 Sycamore Road
216052        68 Spencer Crescent
216053             21 Ochre Close
Name: address1, Length: 216050, dtype: str

In [9]:
epc = normalise.build_address(epc, config.EPC_ADDRESS_COLS)

In [10]:
epc = normalise.build_tokens(epc)

epc['address_tokens']

0                   frozenset({OX4, 10, NEWMAN, ROAD, 3UJ})
1         frozenset({WHEATLEY, 1NL, 10, BARLOW, CLOSE, O...
2                 frozenset({65, OX4, 3PG, ROAD, CAMPBELL})
3         frozenset({6BE, SHIPTON-UNDER-WYCHWOOD, MICHAE...
4         frozenset({OX16, TRAMWAY, CANALSIDE, 5BG, HOUS...
                                ...                        
216049                frozenset({APPLETONS, 7GG, OX12, 30})
216050        frozenset({WELL, WALTON, 1D, OX2, ROAD, 6ED})
216051             frozenset({8, SYCAMORE, OX2, ROAD, 9EJ})
216052         frozenset({OX4, SPENCER, CRESCENT, 68, 4SW})
216053    frozenset({OCHRE, WHEATLEY, 1FQ, CLOSE, 21, OX...
Name: address_tokens, Length: 216050, dtype: object

In [11]:
pp = normalise.strip_whitespace(pp)
pp = normalise.remove_commas(pp, config.PP_ADDRESS_COLS)
pp = normalise.build_address(pp, config.PP_ADDRESS_COLS)
pp = normalise.build_tokens(pp)

In [12]:
from src.cleaning import match

In [13]:
df = match.merge_on_postcode(pp, epc)
df.shape

(2891332, 46)

In [14]:
matched = match.filter_subset_matches(df)
print(df.shape)
print(df['unique_id'].nunique())

(2891332, 46)
133288


In [15]:
filter_subset_matches_measure1 = matched['unique_id'].nunique()

### CHECKPOINT

In [16]:
df = matched.copy()
df = match.drop_ambiguous_matches(df)
print(df.shape)
print(filter_subset_matches_measure1 - df['unique_id'].nunique())

(117362, 46)
975


In [17]:
matches = df.copy()

matches['inspection_to_sale'] = matches['deed_date'] - matches['inspection_date']
drop_neg = matches[matches['inspection_to_sale'] >= '0 days']
grouped = drop_neg.groupby('unique_id')['inspection_to_sale'].transform('min') # minimum epc inspection_to_sale for each sale, transform means that the same index is kept
best = drop_neg[drop_neg['inspection_to_sale'] == grouped] # filtered sales matches to those that equal the min time_to_inspection (inc ties)
print(best[best.duplicated(subset='unique_id', keep=False)]['unique_id'].nunique())
out = drop_neg.sort_values(by=['inspection_to_sale', 'lodgement_date'], ascending=[True, False]).drop_duplicates(subset='unique_id')
print(out.shape)

670
(91150, 47)


Ties found: 670 sales (0.7%) had two or more certificates tied for closest pre-sale inspection; previously chosen arbitrarily, now resolved by latest lodgement_date. EPC features for these sales may differ from Part 4.

In [18]:
df = match.select_presale_epc(df)
df.shape


(91150, 47)

In [19]:
df = match.select_final_columns(df, config.FINAL_COLS, config.RENAME_MAP)
df.shape

(91150, 36)

In [20]:
from src.selection import features

In [21]:
df = features.add_epc_age_band(df, config.AGE_BINS, config.AGE_LABELS)
df = features.add_expired_flag(df, config.EPC_VALIDITY_YEARS)
df = features.add_calendar_features(df)

print(df.shape)

(91150, 43)


In [22]:
df['EPC_expired'].sum()

np.int64(62)

In [23]:
from src.selection import clean

In [24]:
for col, val in config.PLACEHOLDERS.items():
    print(f"{col}:{df[col].isna().sum()}")

print(f"'number_heated_rooms':{df['number_heated_rooms'].isna().sum()}")
print(f"floor height < 1.5m:{(df['floor_height'] < config.THRESHOLDS['min_floor_height']).sum()}")

property_type:8
built_form:2013
tenure:0
'number_heated_rooms':29469
floor height < 1.5m:1427


In [25]:
df = clean.nullify_invalid_values(df, config.PLACEHOLDERS, config.THRESHOLDS['min_floor_height'])
print(df.shape)


(91150, 43)


In [26]:
for col, val in config.PLACEHOLDERS.items():
    print(f"{col}:{df[col].isna().sum()}")

print(f"'number_heated_rooms':{df['number_heated_rooms'].isna().sum()}")
print(f"floor height < 1.5m:{(df['floor_height'] < config.THRESHOLDS['min_floor_height']).sum()}")

property_type:8
built_form:2040
tenure:21458
'number_heated_rooms':29603
floor height < 1.5m:0


In [27]:
df = clean.apply_exclusions(df, config.THRESHOLDS)
print(df.shape)


(91143, 43)


In [28]:
df = clean.impute_floor_height(df)
print(df.shape)
print(df['floor_height_imputed'].sum())

print(df.head())

(91143, 44)
2046
                                    unique_id  price_paid  deed_date  \
1719325  2859C1AC-5B77-52B4-E063-4804A8C05948      870000 2024-04-19   
1435014  2ACACE8D-2773-295E-E063-4804A8C0B0EB      435000 2023-11-15   
719384   0E082197-24E3-5C09-E063-4704A8C0A10E      305000 2023-10-20   
697593   2ACACE8D-280B-295E-E063-4804A8C0B0EB      345000 2023-03-03   
697642   152AB734-15C3-E651-E063-4704A8C061D9      350000 2023-03-03   

                                          address saon paon           street  \
1719325  8 SHIPLAKE BOTTOM PEPPARD COMMON RG9 5HL  NaN    8  SHIPLAKE BOTTOM   
1435014                    25 ANNING WAY OX11 9GR  NaN   25       ANNING WAY   
719384                    8 LOCK CRESCENT OX5 1HE  NaN    8    LOCK CRESCENT   
697593                19 LEMONGRASS ROAD OX27 8BQ  NaN   19  LEMONGRASS ROAD   
697642                21 LEMONGRASS ROAD OX27 8BQ  NaN   21  LEMONGRASS ROAD   

               locality              town           district  ...  \


In [29]:
(df['floor_height'] < 1.5).sum()

np.int64(0)

In [30]:
from src.selection import save

In [31]:
print(df.shape)
print(df.dtypes)
save.save_parquet(df, config.PROCESSED_PATH, config.PROCESSED_FILENAME)

(91143, 44)
unique_id                                 str
price_paid                              int64
deed_date                      datetime64[ns]
address                                   str
saon                                      str
paon                                      str
street                                    str
locality                                  str
town                                      str
district                                  str
postcode                                  str
property_subtype                          str
new_build                                 str
estate_type                               str
transaction_category                      str
certificate_number                        str
built_form                                str
construction_age_band                     str
current_energy_efficiency               int64
current_energy_rating                     str
energy_consumption_current              int64
extension_count       

WindowsPath('C:/Users/Tom/Exercism/python/Week_10_land_registry/data/processed/pp_epc_oxfordshire_2015_2025.parquet')

In [32]:
import pandas as pd
loaded = pd.read_parquet('../data/processed/pp_epc_oxfordshire_2015_2025.parquet')

print(loaded.shape)
print(loaded.dtypes)

(91143, 44)
unique_id                                 str
price_paid                              int64
deed_date                      datetime64[ns]
address                                   str
saon                                      str
paon                                      str
street                                    str
locality                                  str
town                                      str
district                                  str
postcode                                  str
property_subtype                          str
new_build                                 str
estate_type                               str
transaction_category                      str
certificate_number                        str
built_form                                str
construction_age_band                     str
current_energy_efficiency               int64
current_energy_rating                     str
energy_consumption_current              int64
extension_count       